# Bước 3: ANN và 1D-CNN trên Google Colab

Notebook huấn luyện mạng ANN và 1D-CNN theo đúng thiết lập của Walsh et al. 2024. Có 6 cấu hình, mỗi cấu hình chạy với 5 seed, tổng cộng 30 mô hình. Mỗi mô hình lưu dự đoán cho đủ 12.011 phổ. Ô phân tích ở cuối tính RMSEP, so sánh với mốc của Walsh, rồi chạy lại tầng quyết định của Bước 2 bằng dự đoán của CNN.

**Chuẩn bị:**
1. Upload `MangoDMC_NIR_Data_v3.csv` vào thư mục `MyDrive/De_Tai_Nong_San/data/` trên Google Drive.
2. Chọn *Runtime → Change runtime type → T4 GPU*.
3. Chạy lần lượt từng ô. Nếu bước **chạy thử** (mục 5) báo lỗi, dừng lại và gửi thông báo lỗi cho Claude.

**Thời gian:** chưa đo được. Mỗi mô hình in số epoch và số giây khi chạy xong. Nếu Colab ngắt kết nối giữa chừng, chạy lại notebook từ đầu: các mô hình đã xong được lưu trên Drive và sẽ được bỏ qua.

**Kết quả:** được lưu trong `MyDrive/De_Tai_Nong_San/ket_qua_buoc3/`; ô cuối tạo thêm file nén `MyDrive/De_Tai_Nong_San/ket_qua_buoc3.zip`.

## 1. Kiểm tra GPU và TensorFlow

In [ ]:
%env TF_CPP_MIN_LOG_LEVEL=2
!nvidia-smi -L
try:
    import tensorflow as tf
except ImportError:  # in case the Colab image no longer ships TensorFlow
    !pip install -q tensorflow
    import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 2. Kết nối Google Drive

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/De_Tai_Nong_San')
DATA_CSV = DRIVE / 'data' / 'MangoDMC_NIR_Data_v3.csv'
OUT = DRIVE / 'ket_qua_buoc3'
assert DATA_CSV.exists(), f'Chưa thấy file dữ liệu {DATA_CSV}. Upload file vào đó rồi chạy lại ô này.'
OUT.mkdir(parents=True, exist_ok=True)
print('Dữ liệu:', DATA_CSV, '| Kết quả:', OUT)

## 3. Lấy mã nguồn từ GitHub

In [ ]:
import sys

REPO_URL = 'https://github.com/dohieunghia2002/De_Tai_Nong_San.git'
REPO = Path('/content/De_Tai_Nong_San')
if not REPO.exists():
    !git clone -q {REPO_URL} {REPO}
else:
    !git -C {REPO} pull -q
!git -C {REPO} log --oneline -1
link = REPO / 'data' / DATA_CSV.name
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DATA_CSV)
%cd {REPO}
sys.path.insert(0, str(REPO))

## 4. Kiểm tra dữ liệu (dừng với lỗi nếu số phổ hoặc số quả không khớp)

In [ ]:
from src.data import count_table, load_data

print(count_table(load_data()))

## 5. Chạy thử

Mỗi cấu hình chạy 2 epoch trên 1.000 phổ, chỉ để kiểm tra mã nguồn. Kết quả chạy thử để ở `/content/smoke_test`, không ghi lên Drive và không dùng để báo cáo.

In [ ]:
!python -m src.step3_train_nn --smoke --out /content/smoke_test

## 6. Huấn luyện đầy đủ (6 cấu hình × 5 seed)

Mỗi mô hình in ra một dòng gồm số epoch, thời gian và RMSEP sơ bộ. Có thể chạy lại ô này sau khi bị ngắt kết nối.

In [ ]:
!python -m src.step3_train_nn --out "{OUT}"

## 7. Phân tích: RMSEP so với Walsh 2024 và tầng quyết định với CNN

In [ ]:
!python -m src.step3_analysis --runs-dir "{OUT}"

## 8. Đóng gói kết quả

In [ ]:
import zipfile

ZIP = DRIVE / 'ket_qua_buoc3.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.rglob('*')):
        if f.is_file() and f.suffix in ('.csv', '.json'):
            z.write(f, Path('results/step3') / f.relative_to(OUT))
    for name in ('summary.json', 'step3.json', 'step3_regression.csv', 'step3_decision_metrics.csv'):
        if (REPO / 'results' / name).exists():
            z.write(REPO / 'results' / name, Path('results') / name)
print('Đã tạo', ZIP, f'({ZIP.stat().st_size / 1e6:.1f} MB)')

**Bước tiếp theo:** tải `ket_qua_buoc3.zip` từ Drive về máy, giải nén vào thư mục gốc của repo. Thao tác này tạo `results/step3/` và ghi đè `results/summary.json` bằng bản đã có thêm mục `step3`. Sau đó nhờ Claude phân tích kết quả.